### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True # Use 4bit quantization to reduce memory usage.

LLM_MODEL = "llama-2-7b-chat-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/" + LLM_MODEL,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 256, # Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0, # 0 is optimized
    bias = "none",    # "none" is optimized
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,  # rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.524 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


### Dataset

In [ ]:
from datasets import load_dataset

LOCATION = None
if LOCATION is None:
    raise ValueError("Please set the LOCATION variable to the root directory of the project.")

TRAIN_DATASET_NAME = 'lc-quad-1'
dataset = load_dataset("json", data_files=f"{LOCATION}/llm-ft/data/{TRAIN_DATASET_NAME}-train.json")

print(dataset['train'][0])

{'question': 'How many movies did Stanley Kubrick direct?', 'query': 'SELECT DISTINCT COUNT(?uri) WHERE {?uri <http://dbpedia.org/ontology/director> <http://dbpedia.org/resource/Stanley_Kubrick>  . }', 'topic_entities': 'http://dbpedia.org/resource/Stanley_Kubrick (Stanley Kubrick)'}


<a name="Data"></a>
### Data Prep

In [4]:
from unsloth.chat_templates import get_chat_template

# 1. Setup the tokenizer for Llama-2 formatting
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # Standard mapping
)

def formatting_prompts_func(examples):
    questions      = examples["question"]
    topic_entities = examples["topic_entities"]
    gold_queries   = examples["query"]
    
    texts = []
    
    # Iterate through the batch
    for question, entity, gold_query in zip(questions, topic_entities, gold_queries):
        
        # 2. Construct the standard message structure
        user_content = f"Question:\n{question}\n\nTopic Entities:\n{entity}"
        
        conversation = [
            {
                "role": "system",
                "content": "You are a SPARQL expert. Given an input question and topic entities, generate a SPARQL query that retrieves the answer from the Freebase knowledge graph. Ensure that the query is syntactically correct."
            },
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": gold_query
            }
        ]
        
        # 3. Apply the template automatically
        # unsloth automatically adds the correct EOS tokens here
        text = tokenizer.apply_chat_template(conversation, tokenize = False, add_generation_prompt = False)
        
        texts.append(text)
        
    return { "text" : texts }

# Apply to your dataset
dataset = dataset.map(formatting_prompts_func, batched = True)

print(dataset['train'][0])

{'question': 'How many movies did Stanley Kubrick direct?', 'query': 'SELECT DISTINCT COUNT(?uri) WHERE {?uri <http://dbpedia.org/ontology/director> <http://dbpedia.org/resource/Stanley_Kubrick>  . }', 'topic_entities': 'http://dbpedia.org/resource/Stanley_Kubrick (Stanley Kubrick)', 'text': '<s>[INST] <<SYS>>\nYou are a SPARQL expert. Given an input question and topic entities, generate a SPARQL query that retrieves the answer from the Freebase knowledge graph. Ensure that the query is syntactically correct.\n<</SYS>>\n\nQuestion:\nHow many movies did Stanley Kubrick direct?\n\nTopic Entities:\nhttp://dbpedia.org/resource/Stanley_Kubrick (Stanley Kubrick) [/INST] SELECT DISTINCT COUNT(?uri) WHERE {?uri <http://dbpedia.org/ontology/director> <http://dbpedia.org/resource/Stanley_Kubrick>  . } </s>'}


<a name="Train"></a>
### Train the model

In [6]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [7]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 4090. Max memory = 23.524 GB.
6.369 GB of memory reserved.


In [8]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 1 | Total steps = 63
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 639,631,360 of 7,378,046,976 (8.67% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.541400
2,2.443900
3,1.280700
4,0.865400
5,0.797800
6,6.957500
7,0.839400
8,0.531600
9,1.007200
10,0.646700


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained(LLM_MODEL + "_lora_" + TRAIN_DATASET_NAME)
tokenizer.save_pretrained(LLM_MODEL + "_lora_" + TRAIN_DATASET_NAME)

('llama-2-7b-chat-bnb-4bit_lora_lc-quad-1/tokenizer_config.json',
 'llama-2-7b-chat-bnb-4bit_lora_lc-quad-1/special_tokens_map.json',
 'llama-2-7b-chat-bnb-4bit_lora_lc-quad-1/chat_template.jinja',
 'llama-2-7b-chat-bnb-4bit_lora_lc-quad-1/tokenizer.model',
 'llama-2-7b-chat-bnb-4bit_lora_lc-quad-1/added_tokens.json',
 'llama-2-7b-chat-bnb-4bit_lora_lc-quad-1/tokenizer.json')